# Homework 8: Grand Comparison of ML Algorithms & SHAP Interpretation

**F&W ECOL 458 — Environmental Data Science**  
**[YOUR NAME]**  
**Submission:** Upload your completed `.ipynb` to Canvas. **Before submitting, do Runtime > Restart and run all** to ensure all outputs are current.

---

### Overview

In this assignment you will benchmark **every ML algorithm covered in the course** on two tasks — one classification and one regression — and use **SHAP** to interpret the best-performing models. This serves as a comprehensive review of Lectures 14–21.

**Point breakdown:**

| Part | Topic | Points |
|---|---|---|
| 1 | Classification: All algorithms on the Covertype dataset | 30 |
| 2 | Regression: All algorithms on the SW radiation dataset | 30 |
| 3 | SHAP interpretation of best models | 25 |
| 4 | Reflection and algorithm selection | 15 |
| **Total** | | **100** |

### Setup


In [ ]:
# Run this cell first to install required packages
# !pip install xgboost shap -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")


## Part 1: Classification — All Algorithms on the Covertype Dataset (30 points)

The [Covertype dataset](https://archive.ics.uci.edu/ml/datasets/covertype) contains cartographic variables (elevation, aspect, slope, hillshade, soil type, etc.) for 30x30 m cells in the Roosevelt National Forest, Colorado. The target is the forest cover type (7 classes).

Use these feature names throughout:
```python
feat_names = (['Elevation', 'Aspect', 'Slope',
               'Horiz_Dist_Hydro', 'Vert_Dist_Hydro', 'Horiz_Dist_Road',
               'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm',
               'Horiz_Dist_Fire'] +
              [f'Wilderness_{i}' for i in range(1, 5)] +
              [f'Soil_{i}' for i in range(1, 41)])

cover_names = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine",
               "Cottonwood", "Aspen", "Douglas-fir", "Krummholz"]
```


### 1.1 Prepare the data (3 pts)

1. Load the Covertype dataset with `fetch_covtype()`.
2. Subsample to **10,000 samples** using `np.random.RandomState(42)`.
3. Split into 75% train / 25% test with `stratify=y_sub`.
4. Create **scaled** versions of the train/test sets using `StandardScaler` (needed for algorithms that require scaling).
5. Print the number of samples per class in the training set.


In [ ]:
# YOUR CODE HERE




### 1.2 Train all classifiers (12 pts)

Train the following **7 classifiers** on the training data:

| Algorithm | Use scaled data? | Key parameters |
|---|---|---|
| Logistic Regression | Yes | `max_iter=1000` |
| Gaussian Naive Bayes | Yes | default |
| KNN (k=5) | Yes | `n_neighbors=5` |
| SVM (RBF) | Yes | default |
| Decision Tree | No | `max_depth=15` |
| Random Forest | No | `n_estimators=200` |
| XGBoost | No | `n_estimators=200, max_depth=6, learning_rate=0.1` |

For each classifier, record:
- **Test accuracy**
- **Training time** (use `time.time()`)
- Whether it required scaling

Print a summary table and create a **horizontal bar chart** comparing accuracy across all 7 algorithms.

*Hint: For XGBoost classification, use `xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')`.*


In [ ]:
# YOUR CODE HERE




### 1.3 Confusion matrix for the best model (5 pts)

1. Identify which classifier achieved the highest test accuracy.
2. Plot its confusion matrix using `ConfusionMatrixDisplay.from_predictions()` with `cover_names` as labels.

**Answer:**
1. Which classifier performed best?
2. Which two cover types are most commonly confused with each other?
3. Propose an ecological explanation for why those two types are difficult to distinguish.


In [ ]:
# YOUR CODE HERE




*Your written answers here:*



### 1.4 Feature importance comparison (10 pts)

1. Extract **Gini feature importance** from both the Random Forest and XGBoost classifiers.
2. Plot the **top 10 features** for each model as side-by-side horizontal bar charts.
3. Compute **permutation importance** for the best classifier (use `permutation_importance` with `n_repeats=10` on the test set).
4. Plot the top 10 permutation importances with error bars.

**Answer:**
1. What is the single most important feature according to all three methods (RF Gini, XGBoost Gini, permutation)? Does this make ecological sense?
2. Name one feature that ranks high in Gini importance but low in permutation importance (or vice versa). Explain why the two methods might disagree.
3. Based on the importance rankings, which features contribute almost nothing and could be removed?


In [ ]:
# YOUR CODE HERE




*Your written answers here:*



## Part 2: Regression — All Algorithms on the SW Radiation Dataset (30 points)

The SW radiation dataset contains atmospheric and surface variables used to predict incoming shortwave radiation at the Earth's surface:

| Feature | Description | Unit |
|---|---|---|
| SZA | Solar Zenith Angle | degrees |
| AOD | Aerosol Optical Depth | unitless |
| COD | Cloud Optical Depth | unitless |
| CLD_FRAC | Cloud Fraction | 0-1 |
| UW | Water Column | cm |
| TO3 | Total Ozone | DU |
| Pressure | Surface Air Pressure | hPa |
| BSA | Black-Sky Albedo | 0-1 |
| WSA | White-Sky Albedo | 0-1 |

Targets: **SW_direct** (direct beam), **SW_diffuse** (scattered), and **SW = SW_direct + SW_diffuse** (total).


### 2.1 Load and explore the data (3 pts)

1. Load SW.csv (or generate synthetic data using the code from Lecture 20/21 if the file is not available).
2. Compute `SW = SW_direct + SW_diffuse`.
3. Print the dataset shape and `df.describe()`.
4. Create scatter plots of **SZA vs. SW**, **CLD_FRAC vs. SW_direct**, and **AOD vs. SW_diffuse**.

**Answer:** Based on the scatter plots, which relationships appear linear and which appear nonlinear?


In [ ]:
# YOUR CODE HERE




*Your written answer here:*



### 2.2 Train all regressors (12 pts)

Using `SW` (total shortwave) as the target, train the following **6 regressors**:

| Algorithm | Use scaled data? | Key parameters |
|---|---|---|
| Linear Regression | Yes | default |
| KNN (k=10) | Yes | `n_neighbors=10` |
| SVR (RBF) | Yes | `C=100` |
| Decision Tree | No | `max_depth=10` |
| Random Forest | No | `n_estimators=200` |
| XGBoost | No | `n_estimators=500, max_depth=6, learning_rate=0.1` |

For each, record:
- **Test R$^2$**
- **Test RMSE**
- **Training time**

Print a summary table and create:
1. A **bar chart comparing Test R$^2$** across all 6 models.
2. A **2x3 grid of 1:1 scatter plots** (observed vs. predicted) for all 6 models, with the R$^2$ value in each panel's title.


In [ ]:
# YOUR CODE HERE




### 2.3 Predict all three radiation components (5 pts)

Using **Random Forest** (or your best-performing model):

1. Train separate models for **SW_direct**, **SW_diffuse**, and **SW_total**.
2. Create a **1x3 panel of 1:1 plots** showing observed vs. predicted for each component.
3. Report the R$^2$ and RMSE for each.

**Answer:** Which radiation component is hardest to predict? Why might this be the case physically?


In [ ]:
# YOUR CODE HERE




*Your written answer here:*



### 2.4 Gini and permutation importance (10 pts)

1. Extract **Gini feature importance** from both Random Forest and XGBoost for the `SW_total` target.
2. Plot side-by-side horizontal bar charts.
3. Compute **permutation importance** for the best model.

**Answer:**
1. Which feature dominates the importance ranking? Is this physically expected?
2. How does the importance ranking differ between SW_direct and SW_diffuse models? (You can reuse models from 2.3.)
3. For the SW_direct model, explain *why* CLD_FRAC should rank high and *why* COD might rank lower than you'd expect.


In [ ]:
# YOUR CODE HERE




*Your written answers here:*



## Part 3: SHAP Interpretation (25 points)

Use `shap.TreeExplainer` on your best **tree-based model** (Random Forest or XGBoost). Install SHAP with `!pip install shap -q` if needed.


### 3.1 SHAP for classification — Covertype (10 pts)

1. Compute SHAP values for the XGBoost classifier on the **test set**.
2. Produce a **SHAP beeswarm (summary) plot**. *(Note: for multi-class, SHAP returns a list of arrays — one per class. Pick the class with the most samples, e.g., Lodgepole Pine, and plot its SHAP values.)*
3. Produce a **SHAP bar plot** (mean |SHAP value| per feature).
4. Produce **SHAP dependence plots** for the top 2 features.

**Answer:**
1. According to the beeswarm plot, does high Elevation increase or decrease the probability of the class you selected? Is this ecologically reasonable?
2. In the dependence plot for Elevation, is the relationship linear or nonlinear? What does the interaction color reveal?
3. Name one feature where the SHAP beeswarm reveals something that the Gini importance bar chart from Part 1.4 could NOT show (e.g., direction of effect, nonlinearity, interaction).


In [ ]:
# YOUR CODE HERE




*Your written answers here:*



### 3.2 SHAP for regression — SW radiation (15 pts)

1. Compute SHAP values for the XGBoost regressor (SW_total target) on the test set.
2. Produce a **SHAP beeswarm plot**.
3. Produce **SHAP dependence plots** for **SZA**, **CLD_FRAC**, and **AOD**.

**Answer:**
1. In the beeswarm plot, what is the direction of effect for each of the top 3 features? (i.e., does high feature value increase or decrease SW?)
2. In the SZA dependence plot, describe the shape of the relationship. Does it match what you know about the cosine of solar zenith angle?

In [ ]:
# YOUR CODE HERE




*Your written answers here:*



## Part 4: Reflection and Algorithm Selection (15 points)


### Algorithm selection guide (8 pts)

Based on your benchmark results from Parts 1–2 and everything you've learned in Lectures 14–21, fill in the following table:

| Scenario | Best algorithm | Why? (1-2 sentences) |
|---|---|---|
| Classify land cover from 6 Landsat bands with 5,000 training samples | | |
| Predict daily GPP from meteorological variables at a flux tower | | |
| Quick baseline to check if a dataset has any predictive signal at all | | |
| Rank which environmental variables most strongly control species distribution | | |
| Cluster satellite pixels into spectral groups with no labeled training data | | |
| You have only 50 labeled training samples and need a reliable classifier | | |
| You need the absolute highest accuracy and have access to a GPU | | |
| You need to explain to a park manager *why* a certain area was classified as high fire risk | | |


*Fill in the table above.*

